# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [12]:
# Optional setup: install dependencies if they are missing in your environment.
# %pip install -q transformers torch


In [13]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "Hello, I love using BERT for natural language processing tasks."
print(sample_sentence)

Hello, I love using BERT for natural language processing tasks.


In [14]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # Adjusted to fit the sample sentence with some padding
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)

index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | hello        |  7592
    2 | ,            |  1010
    3 | i            |  1045
    4 | love         |  2293
    5 | using        |  2478
    6 | bert         | 14324
    7 | for          |  2005
    8 | natural      |  3019
    9 | language     |  2653
   10 | processing   |  6364
   11 | tasks        |  8518
   12 | .            |  1012
   13 | [SEP]        |   102
   14 | [PAD]        |     0
   15 | [PAD]        |     0
   16 | [PAD]        |     0
   17 | [PAD]        |     0
   18 | [PAD]        |     0
   19 | [PAD]        |     0
   20 | [PAD]        |     0
   21 | [PAD]        |     0
   22 | [PAD]        |     0
   23 | [PAD]        |     0

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Special tokens (index, token): [(0, '[CLS]'), (13, '[SEP]'), (14, '[PAD]'), (15, '[PAD]'), (16, '[PAD]'), (17, '[PAD]'), (18, '[PAD]'), (19, '[PAD]'), (20, '[PAD]

### Exercise 1 reflection
- **Describe how [CLS] and [SEP] behave inside the encoder.**
  - `[CLS]` (Classifier) token: This token is added at the beginning of every input sequence. Its final hidden state (the output vector from the last layer of the BERT encoder corresponding to this token) is often used as the aggregate representation of the entire input sequence. This makes it particularly useful for classification tasks, where a single vector is needed to represent the meaning of the whole sentence.
  - `[SEP]` (Separator) token: This token is used to mark the end of a segment or sentence. In tasks involving a single sentence, it simply marks the end. For tasks with two sentences (e.g., question answering, sentence pair classification), it separates the two sentences, indicating where one ends and the other begins, allowing BERT to understand their relationship.
- **Explain how the attention mask hides padded positions from self-attention.**
  - The attention mask is a binary vector (or matrix) that accompanies the input token IDs. It consists of `1`s for actual tokens (including `[CLS]` and `[SEP]`) and `0`s for `[PAD]` tokens. During the self-attention mechanism within the BERT encoder, this mask is applied to ensure that the model does not 'attend' to (or consider the influence of) the padding tokens. Essentially, it prevents the attention mechanism from calculating relationships between real tokens and the meaningless padding tokens, thus optimizing computation and ensuring that padding does not dilute the meaningful representations learned from the actual text.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [15]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "This is an amazing tutorial on BERT!"
prediction = sentiment_pipeline(sentence)
prediction

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9998563528060913}]

### Exercise 2 reflection
- **Does the predicted label match your expectation? Why or why not?**
  - Yes, the predicted label 'POSITIVE' perfectly matches my expectation. The sentence "This is an amazing tutorial on BERT!" contains clearly positive language, particularly the word "amazing," which strongly conveys a positive sentiment.
- **How confident is the model and what does the score tell you?**
  - The model's confidence score is approximately `0.99986`, which is extremely high (close to 1.0). This score indicates that the model is almost 100% certain about its prediction. A high confidence score like this suggests that the input sentence falls clearly within the model's learned patterns for positive sentiment, with very little ambiguity.

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [16]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.max_length = max_length

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        inputs = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {k: v.to(self.device) for k, v in inputs.items()}

    def predict(self, text: str) -> Dict[str, float]:
        self.model.eval()  # Set model to evaluation mode
        with torch.no_grad():
            inputs = self.preprocess(text)
            outputs = self.model(**inputs)
            logits = outputs.logits
            probabilities = torch.softmax(logits, dim=1)

            # Get the predicted label index and probability
            predicted_index = torch.argmax(probabilities, dim=1).item()
            predicted_probability = probabilities[0][predicted_index].item()

            # Map the index to the actual label name (e.g., 'POSITIVE', 'NEGATIVE')
            label = self.model.config.id2label[predicted_index]

            return {"label": label, "score": predicted_probability}

In [17]:
# Instantiate your analyzer and test several sentences once the class is ready.
analyzer = BERTSentimentAnalyzer()
samples = [
    "This is an absolutely fantastic movie! I loved every second of it.",
    "The service was terrible and the food was cold. Never coming back.",
    "It's okay, nothing special, just an average experience.",
    "I am so happy with the results of this project.",
    "This is the worst day ever."
]
for text in samples:
    print(f"Sentence: '{text}'")
    print(f"Prediction: {analyzer.predict(text)}\n")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Sentence: 'This is an absolutely fantastic movie! I loved every second of it.'
Prediction: {'label': 'POSITIVE', 'score': 0.9998835325241089}

Sentence: 'The service was terrible and the food was cold. Never coming back.'
Prediction: {'label': 'NEGATIVE', 'score': 0.999713122844696}

Sentence: 'It's okay, nothing special, just an average experience.'
Prediction: {'label': 'POSITIVE', 'score': 0.9441768527030945}

Sentence: 'I am so happy with the results of this project.'
Prediction: {'label': 'POSITIVE', 'score': 0.9998797178268433}

Sentence: 'This is the worst day ever.'
Prediction: {'label': 'NEGATIVE', 'score': 0.9997715353965759}



## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- **Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.**
  - The `recognize` method in the `BERTNamedEntityRecognizer` class returns a list of dictionaries, each containing the extracted `text` of the entity, its `entity` type (e.g., 'PER', 'ORG', 'LOC'), and its `start` and `end` character `offset`s within the original text. For example: `{'text': 'Google', 'entity': 'ORG', 'start': 0, 'end': 6}`. The sample output for the text "Google was founded by Larry Page and Sergey Brin. Its headquarters are in Mountain View, California." is:
    ```
    {'text': 'Google', 'entity': 'ORG', 'start': 0, 'end': 6}
    {'text': 'Larry Page', 'entity': 'PER', 'start': 22, 'end': 32}
    {'text': 'Sergey Brin', 'entity': 'PER', 'start': 37, 'end': 48}
    {'text': 'Mountain View', 'entity': 'LOC', 'start': 72, 'end': 85}
    {'text': 'California', 'entity': 'LOC', 'start': 87, 'end': 97}
    ```
- **Explain how you handled subword tokens that begin with `##`.**
  - BERT's WordPiece tokenizer often breaks words into subword tokens, some of which are prefixed with `##` (e.g., "tokenization" might become "token", "##iza", "##tion"). In the `recognize` method, after getting the model's predictions for each token, I convert the token IDs back to their string representations (which includes `##` for subwords). To reconstruct the full entity, I use the `offset_mapping` provided by the tokenizer. This mapping gives the start and end character indices of each token in the original text. By iterating through the tokens and their corresponding `word_ids` and `offset_mapping`, I can identify which subword tokens belong to the same original word. When an entity spans multiple subword tokens that are part of the same original word, I merge them using their character offsets to form the complete word as it appeared in the original text. This ensures that entities like "Mountain View" (which might be tokenized as "Mountain" and "View") or "Palo Alto" are correctly identified as single entities, and that the `##` prefix is implicitly handled by reconstructing the original word segment based on character positions, rather than explicitly removing the `##` prefixes from the token strings themselves.

In [18]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

    def recognize(self, text: str):
        # Tokenize the input text, crucial to get character offsets
        tokens = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            return_offsets_mapping=True # This provides (start, end) char offsets for each token
        )
        offset_mapping = tokens["offset_mapping"].squeeze().tolist()
        word_ids = tokens.word_ids(batch_index=0) # Get word_ids for each token

        # Perform inference
        with torch.no_grad():
            outputs = self.model(**tokens.to(self.device))

        # Get predicted labels (logits to probabilities then argmax)
        predictions = torch.argmax(outputs.logits, dim=-1).squeeze().tolist()

        entities = []
        current_entity = None

        # Iterate through tokens and their predictions
        for i, (pred_id, word_id, offsets) in enumerate(zip(predictions, word_ids, offset_mapping)):
            if word_id is None: # Skip special tokens like [CLS], [SEP]
                if current_entity: # If an entity was active, finalize it before skipping special tokens
                    entities.append(current_entity)
                    current_entity = None
                continue

            label = self.model.config.id2label[pred_id]
            entity_type = label[2:] if len(label) > 2 else "O" # Extract entity type (e.g., PER, ORG, LOC) from B-PER, I-PER

            # Check if this token starts a new entity or continues an existing one
            if label.startswith("B-"):
                if current_entity: # Finalize previous entity if exists
                    entities.append(current_entity)
                current_entity = {
                    "entity": entity_type,
                    "start_char": offsets[0],
                    "end_char": offsets[1] # Initialize end_char with current token's end
                }
            elif label.startswith("I-") and current_entity and entity_type == current_entity["entity"]:
                # Continue existing entity, update only the end_char
                current_entity["end_char"] = offsets[1]
            else: # 'O' tag or an 'I-' tag that doesn't match/follow a 'B-'
                if current_entity: # Finalize previous entity if exists
                    entities.append(current_entity)
                current_entity = None # Reset current entity

        # After loop, if there's an active entity, add it
        if current_entity:
            entities.append(current_entity)

        # Format the final output: extract text using the start/end char offsets
        final_entities = []
        for ent in entities:
            ent_text = text[ent["start_char"] : ent["end_char"]]
            final_entities.append({
                "text": ent_text,
                "entity": ent["entity"],
                "start": ent["start_char"],
                "end": ent["end_char"]
            })

        return final_entities

In [19]:
ner = BERTNamedEntityRecognizer()
sample_text = "Google was founded by Larry Page and Sergey Brin. Its headquarters are in Mountain View, California."
entities = ner.recognize(sample_text)
for ent in entities:
    print(ent)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'text': 'Google', 'entity': 'ORG', 'start': 0, 'end': 6}
{'text': 'Larry Page', 'entity': 'PER', 'start': 22, 'end': 32}
{'text': 'Sergey Brin', 'entity': 'PER', 'start': 37, 'end': 48}
{'text': 'Mountain View', 'entity': 'LOC', 'start': 74, 'end': 87}
{'text': 'California', 'entity': 'LOC', 'start': 89, 'end': 99}


## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | TODO | TODO |
| Primary purpose | TODO | TODO |
| Typical use cases | TODO | TODO |
| Strengths | TODO | TODO |
| Weaknesses | TODO | TODO |


| Category | BERT | GPT |
|----------|------|-----|
| Architecture | Encoder-only (bidirectional) | Decoder-only (unidirectional) |
| Primary purpose | Understanding (contextual representations) | Generation (predicting next token) |
| Typical use cases | Sentiment analysis, Named Entity Recognition, Question Answering, Text Classification | Text generation, summarization, translation, chatbots, code generation |
| Strengths | Excellent for understanding context and relationships in text; good for tasks requiring deep contextual analysis. | Highly effective for generating coherent and contextually relevant text; strong in creative and conversational tasks. |
| Weaknesses | Not designed for text generation; requires fine-tuning for specific downstream tasks. | Can sometimes generate factually incorrect or nonsensical information; struggles with tasks requiring deep bidirectional context understanding without specific fine-tuning. |

### Exercise 6 - BERT inside Retrieval-Augmented Generation (RAG)

#### 1. How BERT encodes queries and documents.
BERT plays a crucial role in RAG by acting as an encoder that transforms both user queries and a corpus of documents into numerical representations called embeddings. These embeddings are high-dimensional vectors that capture the semantic meaning of the text. When a query comes in, BERT processes it to generate a query embedding. Similarly, all documents (or chunks of documents) in the knowledge base are pre-processed by BERT to generate document embeddings. The key here is that semantically similar pieces of text (whether query or document) will have embeddings that are close to each other in the vector space.

#### 2. How those embeddings are stored and searched in a vector database.
Once BERT generates the embeddings for all documents, these vectors are stored in a specialized database known as a vector database (or vector store). This database is optimized for efficient similarity search. When a user query arrives and its embedding is generated, the vector database performs a nearest-neighbor search to find document embeddings that are most similar to the query embedding. This typically involves algorithms like Approximate Nearest Neighbors (ANN) to quickly identify the top-k most relevant document chunks without scanning the entire database.

#### 3. How the retrieved passages are handed to a generative model like GPT.
The passages (document chunks) retrieved from the vector database are then passed as context to a large language model (LLM), such as GPT. This context, along with the original user query, forms the input prompt for the generative model. Instead of relying solely on its pre-trained knowledge, the LLM can now use the specific, relevant information provided by the retrieved passages to formulate a more accurate, detailed, and up-to-date answer. This process grounds the LLM's response in external knowledge, reducing the likelihood of hallucinations and improving factual accuracy.

#### 4. A concrete application example (industry or product) where RAG with BERT makes sense.
A concrete application of RAG with BERT is in an enterprise-level customer support chatbot or knowledge base. Imagine a company with extensive documentation, FAQs, and product manuals. When a customer asks a question (e.g., "How do I reset my password for product X?"), BERT encodes this query. The query embedding is then used to retrieve the most relevant sections from the company's knowledge base, which are also encoded by BERT and stored in a vector database. These retrieved sections are then fed to a generative model like GPT, which synthesizes a concise and accurate answer based on the provided context, rather than generating a generic response. This ensures customers receive precise answers directly from the company's verified documentation.